# Project 1 — Customer Segmentation

**Goal:** Discover groups in customer-like data using K-Means clustering and silhouette scores.

Run the cells from top to bottom. Data is generated locally or loaded from scikit-learn; no CSV download is required.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, silhouette_score, mean_absolute_error, mean_squared_error,
    precision_score, recall_score
)
from sklearn.datasets import load_breast_cancer, make_classification, make_blobs
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
print("Setup complete")


## Project 1 — Customer Segmentation (K-Means + Silhouette Score)
**Goal:** Discover groups in customer-like data without labels.

In [ ]:
# Create a reproducible synthetic customer dataset
X_customers, _ = make_blobs(
    n_samples=600, centers=4, n_features=5, cluster_std=1.7,
    random_state=RANDOM_STATE
)
customer_df = pd.DataFrame(X_customers, columns=[f"feature_{i+1}" for i in range(5)])
display(customer_df.head())

In [ ]:
# Compare cluster counts using silhouette score
scores = {}
for k in range(2, 8):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(
        StandardScaler().fit_transform(customer_df)
    )
    scores[k] = silhouette_score(StandardScaler().fit_transform(customer_df), labels)

plt.figure(figsize=(7, 4))
plt.plot(list(scores.keys()), list(scores.values()), marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Choosing k for customer segmentation")
plt.grid(True)
plt.show()

best_k = max(scores, key=scores.get)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_df)
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
customer_df["segment"] = kmeans.fit_predict(X_scaled)
print("Selected k:", best_k)
display(customer_df.groupby("segment").mean().round(2))

In [ ]:
# Visualize two features with segment labels
plt.figure(figsize=(7, 5))
for segment in sorted(customer_df["segment"].unique()):
    part = customer_df[customer_df["segment"] == segment]
    plt.scatter(part["feature_1"], part["feature_2"], label=f"Segment {segment}", alpha=0.7)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Customer segments (first two features)")
plt.legend()
plt.show()

## Possible extensions
- Compare K-Means with DBSCAN and Gaussian Mixture Models.
- Profile each cluster using meaningful domain-specific features.